# Forecast initial-condition soil-moisture errors: process

For each forecast group (Jan/Jun 2012) and experiment, compute the top-10 cm soil-moisture ensemble mean, spread, bias, t-test significance, PCC and RMSE against ESA CCI on the forecast start date. One NetCDF per group is written to `data/fcst_lnd_model_evaluation/initial_condition_error/` and plotted by notebook 04. Existing products are skipped unless `FORCE_COMPUTE = True`.

## 1. Settings

In [1]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")   # observations (shared archive)
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")   # land mask, soil-layer thickness
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "fcst_lnd_model_evaluation"

FORECAST_GROUPS = {
    "Jan2012": {"models": ["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"], "period": "201201-201203",
                "init_time": "2012-01-01"},
    "Jun2012": {"models": ["CTRL10-S1", "CAPT10-S1", "DART40-S1"], "period": "201206-201208",
                "init_time": "2012-06-01"},
}
OBS_KEY = "ESA_CCI"
VARIABLE_MAP = {"SOILWATER_10CM": "H2OSOI"}   # model variable -> observation variable
DEPTH_CM = 10.0
MODEL_IS_INTEGRATED = True   # SOILWATER_10CM is already a 0-10 cm total (kg m-2)
ALPHA = 0.05
SIGNIFICANCE_THRESHOLD = None
SIG_MIN_N = 5
SIG_EPS_STD = 1e-12
NEAREST_TOLERANCE = "12H"
FORCE_COMPUTE = False
# File collection: forecast ("fc") daily land time series, files <var>.EN##.<year>.nc
RUN, FREQ = "fc", "daily"
MODEL_TEMPLATE, ENS_PREFIX, ENS_WIDTH, ENS_START = "%(var)s.%(ens)s.%(year)d.nc", "EN", 2, 1

## 2. Project setup

In [2]:
import sys
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(WORKFLOW_NAME, output_root=DIAGNOSTIC_OUTPUT_ROOT)

import pandas as pd
from configs.initial_land_experiment_registry import build_experiments
from configs.initial_land_observation_registry import get_obs_registry, get_obs_file
from util.initial_lnd_file_collection import FileCollectionConfig, S2SFileCollector
from util.initial_lnd_soil_error import SoilBiasFromFilesConfig, SoilBiasAssessorFromCollectedFiles

OUT_DIR = WORKFLOW_DATA_DIR / "initial_condition_error"
OUT_DIR.mkdir(parents=True, exist_ok=True)
collector = S2SFileCollector(
    exp_list=FORECAST_GROUPS, exp_dict=build_experiments(INPUT_DATA_ROOT),
    obs_registry=get_obs_registry(OBSERVATION_ROOT),
    s2s_var_dict=VARIABLE_MAP, get_obs_file_func=get_obs_file,
)

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 3. Process and cache

In [3]:
for group, meta in FORECAST_GROUPS.items():
    init_time = pd.to_datetime(meta["init_time"])
    pending = {var: OUT_DIR / f"soil_bias_{group}_{FREQ}_{RUN}_{OBS_KEY}_{var}_{init_time:%Y%m%d}.nc"
               for var in VARIABLE_MAP}
    pending = {var: path for var, path in pending.items() if FORCE_COMPUTE or not path.exists()}
    if not pending:
        print(f"[{group}] all products exist; skipping")
        continue
    files = collector.collect(
        FileCollectionConfig(group=group, freq=FREQ, run=RUN, obs=OBS_KEY, period=meta["period"],
                             model_template=MODEL_TEMPLATE, ens_prefix=ENS_PREFIX,
                             ens_width=ENS_WIDTH, ens_start=ENS_START),
        vars_to_process=list(pending), verbose=True,
    )
    for model_var, out_nc in pending.items():
        if model_var not in files:
            continue
        assessor = SoilBiasAssessorFromCollectedFiles(SoilBiasFromFilesConfig(
            out_dir=OUT_DIR, landmask_file=LAND_SUPPORT_DIR / "landmask_1x1.nc",
            soilayer_file=LAND_SUPPORT_DIR / "dzsoi_elm.nc",
            model_var_name=model_var, obs_var_name=VARIABLE_MAP[model_var], out_var_name=model_var,
            model_is_integrated=MODEL_IS_INTEGRATED, depth_cm=DEPTH_CM, alpha=ALPHA,
            significance_threshold=SIGNIFICANCE_THRESHOLD, sig_min_n=SIG_MIN_N, sig_eps_std=SIG_EPS_STD,
            nearest_tolerance=NEAREST_TOLERANCE,
        ))
        ds = assessor.compute_one_date_from_varfiles(files[model_var], target_date=init_time, obs_key=OBS_KEY)
        print(f"[{group}] Saved: {assessor.save(ds, out_nc, overwrite=FORCE_COMPUTE)}")

[Jan2012] all products exist; skipping
[Jun2012] all products exist; skipping
